---
# PARTIE 2 -- Structured Streaming (après-midi)

## 2.1 Concepts fondamentaux

### Du batch au streaming

Le batch traite un ensemble de données **figé** : on sait à l'avance combien
il y a de lignes, on peut trier, on peut faire des jointures complètes.

Le streaming traite un flux de données **continu** : les données arrivent
en permanence, on ne connaît pas la fin, et certains événements peuvent
arriver en retard.

Spark Structured Streaming répond à ce problème avec un modèle élégant :
le flux est traité comme une **table infinie** qui s'agrandit au fil du temps.
Les requêtes SQL ou DataFrame s'y appliquent sans modification.

```
Flux d'entrée  :  [evt1] [evt2] [evt3] ...  [evtN]  ...
                     |      |      |              |
                  +--+------+------+--------------+----→ temps
                  |
                  Spark : exécution de micro-batchs toutes les X secondes
                  |
                  Résultat : table de résultats mise à jour en continu
```

### Vocabulaire

| Terme | Définition |
|-------|-----------|
| **Trigger** | Fréquence d'exécution des micro-batchs |
| **Checkpoint** | Sauvegarde de l'état pour la reprise après panne |
| **Watermark** | Délai maximal toléré pour les données tardives |
| **Fenêtre glissante** | Agrégation sur une plage de temps mobile |
| **Sink** | Destination de l'écriture (console, fichier, Delta...) |


---
## 2.2 Le simulateur de flux

En production, le flux proviendrait de Kafka ou de l'API GBFS en direct.
Pour ce cours, un script Python **rejoue les données historiques** en écrivant
des fichiers JSON dans un répertoire surveillé par Spark.

Ce mécanisme -- la **file source** (file source) -- est le moyen le plus simple
de tester Structured Streaming sans infrastructure externe.

> Le simulateur est fourni dans `scripts/simulateur_flux.py`.
> Lancez-le dans un terminal séparé **avant** d'exécuter les cellules suivantes.
>
> ```bash
> python scripts/simulateur_flux.py --output data/output/stream_input --vitesse 3
> ```
>
> L'option `--vitesse 3` signifie que chaque seconde réelle correspond à
> 3 minutes de données historiques.


In [10]:
# Vérification : le simulateur tourne-t-il ?
import time
from pathlib import Path

def compter_fichiers_stream(attente_sec: int = 5) -> list[Path]:
    """Attend et retourne les fichiers JSON produits par le simulateur."""
    time.sleep(attente_sec)
    return sorted(
        Path(STREAM_SOURCE_DIR).glob("*.json"),
        key=lambda p: p.stat().st_mtime,
    )

fichiers = compter_fichiers_stream(3)
n = len(fichiers)
if n == 0:
    print("[ATTENTION] Aucun fichier JSON trouvé dans le répertoire de streaming.")
    print("  Vérifiez que le simulateur tourne dans un terminal séparé.")
    print(f"  Répertoire surveillé : {STREAM_SOURCE_DIR}")
else:
    dernier = fichiers[-1]
    print(f"[OK] {n} fichier(s) JSON détecté(s).")
    print(f"  Dernier fichier : {dernier.name}")
    print("  Aperçu :")
    print(dernier.read_text(encoding="utf-8")[:400])


[OK] 21 fichier(s) JSON détecté(s).
  Dernier fichier : snapshot_000021.json
  Aperçu :
[{"station_id": 213688169, "nom_station": "Benjamin Godard - Victor Hugo", "code_arr": 213688, "capacite": 35, "velos_meca": 2, "velos_elec": 1, "bornettes_libres": 32, "horodatage": "2020-11-26T14:25Z"}, {"station_id": 123095125, "nom_station": "Alibert - Jemmapes", "code_arr": 123095, "capacite": 60, "velos_meca": 2, "velos_elec": 0, "bornettes_libres": 58, "horodatage": "2020-11-26T14:25Z"}, {"


---
## 2.3 Source de streaming : `readStream`

`spark.readStream` fonctionne exactement comme `spark.read`, avec deux différences :

1. Il retourne un **DataFrame de streaming** (pas un DataFrame ordinaire).
2. Il ne peut pas être affiché directement avec `.show()` -- il faut déclencher
   une **requête de streaming** avec `.writeStream`.


In [11]:
# Schéma du flux JSON produit par le simulateur
schema_flux = StructType([
    StructField("station_id",       IntegerType(),   False),
    StructField("nom_station",      StringType(),    True),
    StructField("code_arr",         IntegerType(),   True),
    StructField("capacite",         IntegerType(),   True),
    StructField("velos_meca",       IntegerType(),   True),
    StructField("velos_elec",       IntegerType(),   True),
    StructField("bornettes_libres", IntegerType(),   True),
    StructField("horodatage",       TimestampType(), False),   # timestamp parsé
])

# Création du DataFrame de streaming
# au plus 2 fichiers par micro-batch
stream_df = (
# TODO lire le flux du simulateur
 )

print(f"Est un streaming DataFrame : {stream_df.isStreaming}")
print(f"Colonnes : {stream_df.columns}")
# On ne peut pas appeler .count() ou .show() sur un streaming DataFrame


NameError: name 'StructType' is not defined

In [ ]:
# Ajout des colonnes calculées -- exactement comme en batch
stream_enrichi = (
    stream_df # TODO Ajouter les calculs
)
print("Colonnes après enrichissement :", stream_enrichi.columns)


---
## 2.4 Première requête : sink console

La sortie `console` écrit les résultats dans le terminal Jupyter.
C'est utile pour le débogage -- jamais pour la production.


In [ ]:
# Requête de streaming vers la console
# outputMode :
#   "append"  -- écrit uniquement les nouvelles lignes (défaut pour les flux sans agrégation)
#   "complete" -- réécrit l'intégralité du résultat à chaque batch
#   "update"  -- écrit uniquement les lignes qui ont changé
#
# La fonction start() lance le streaming

q_console = (
    stream_enrichi # Ecrire dans la console "station_id", "nom_station", "code_arr", "horodatage", "velos_total", "taux_occupation", "est_vide"
                   # toutes les 5 secondes en mode "append", 10 lignes
)

# La requête tourne en arrière-plan.
# awaitTermination() bloquerait indéfiniment -- on attend juste quelques batchs.
time.sleep(20)
print(f"Statut de la requête : {q_console.status}")
print(f"Batchs traités : {q_console.lastProgress['numInputRows'] if q_console.lastProgress else 'N/A'}")
q_console.stop()
print("Requête console arrêtée.")


---
## 2.5 Agrégations sur fenêtres temporelles glissantes

L'agrégation sur des **fenêtres temporelles** est la requête streaming la plus
courante en pratique. Elle répond à des questions du type :
"Combien de vélos sont disponibles par arrondissement sur les 10 dernières minutes ?"

Spark propose deux types de fenêtres temporelles :

| Type | Définition | Exemple |
|------|-----------|---------|
| Fenêtre **basculante** | Fenêtres fixes, sans chevauchement | 0-10 min, 10-20 min... |
| Fenêtre **glissante** | Fenêtres qui se chevauchent | [0-10], [5-15], [10-20]... |

```
Fenêtre basculante (10 min)  : |--w1--|--w2--|--w3--|
Fenêtre glissante (10/5)   : |--w1--|          Taille=10, pas=5
                                  |--w2--|
                                       |--w3--|
```


In [ ]:
from pyspark.sql.functions import window

# Agrégation glissante : disponibilité moyenne par arrondissement
# Fenêtre de 10 minutes, avançant toutes les 2 minutes
df_fenetre_arr = (
    stream_enrichi # requête en syntaxe fonctionnelle
)

print("Schéma de la requête fenêtrée :")
df_fenetre_arr.printSchema()


In [ ]:
# Écriture vers Delta Lake (sink delta) -- mode append
q_fenetres = (
    df_fenetre_arr # ?
)

print(f"Requête démarrée : {q_fenetres.name}")
print(f"ID               : {q_fenetres.id}")
print("En attente de 3 batchs...")
time.sleep(35)
print(f"Dernier progrès : {q_fenetres.lastProgress}")


In [ ]:
# Lecture des résultats accumulés dans Delta
path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")

try:
    # TODO Lire les données au format delta, trier par début de fenêtre et en afficher 30
except Exception as e:
    print(f"Pas encore de données : {e}")
    print("Attendez encore quelques secondes et relancez cette cellule.")


---
## 2.6 Alertes : détection de stations en rupture prolongée

L'objectif opérationnel est d'alerter l'équipe de maintenance quand une station
reste vide (zéro vélo disponible) pendant au moins **deux fenêtres consécutives**,
soit 20 minutes de rupture continue.

### Approche : `foreachBatch`

Pour une logique d'alerte complexe (état entre batchs, seuil consécutif),
`foreachBatch` est plus adapté que les agrégations pures. Il permet d'exécuter
une fonction Python arbitraire sur chaque micro-batch.


In [ ]:
# Table d'alertes : schema
schema_alertes = StructType([
    StructField("station_id",   IntegerType(),   False),
    StructField("nom_station",  StringType(),    True),
    StructField("code_arr",     IntegerType(),   True),
    StructField("debut_rupture",TimestampType(), True),
    StructField("fin_rupture",  TimestampType(), True),
    StructField("duree_min",    IntegerType(),   True),
    StructField("ts_alerte",    TimestampType(), True),
])

# Initialisation de la table Delta d'alertes (vide)
# TODO Créer la table () partir d'un DataFrame)
print(f"Table d'alertes initialisée : {DELTA_ALERTES}")


In [ ]:
from delta.tables import DeltaTable

# Mémoire inter-batchs : stations actuellement en rupture et depuis quand
etat_ruptures: dict[int, dict] = {}

def traiter_batch_alertes(batch_df, batch_id: int) -> None :
    """Fonction appelée par foreachBatch pour chaque micro-batch.

    Détecte les stations qui passent en rupture ou sortent de rupture.
    Écrit les alertes complètes dans la table Delta.

    Args:
        batch_df : DataFrame du micro-batch courant.
        batch_id : Identifiant séquentiel du batch.
    """
    global etat_ruptures

    # TODO : Sortir di `batch_df` est vide

    # Agrégation au niveau station sur ce batch
    # On souhaite collecter :
    # - l'horodatage le plus ancien
    # - l'horodatage le plus récent
    # - le nombre d'observations où la station est vide
    # - le nombre total de stations
    # - le code de l'arrondissement
    # - le nom de la station
    etat_batch = (
        batch_df # ?
    ). # ??

    alertes = []
    # Examiner les transitions :
    # - non vide -> vide
    # - vide -> non vide
    # Dans le premier cas, ajouter la les informations dela station dans `etat_rupture`.
    # Dans le second cas :
    # - calculer le temps pendant le quel la station a été en rupture
    # - si ce temps est > 10 minutes, ajouter une alerte avec les informations détailllées
    # - retirer la station de la liste `etat_rupture`


    # Si des alertes ont été enregistrées,
    # les sauvegarder au format `delta` dans DELTA_ALERTES
    # Laisser une trace dans les logs


In [ ]:
# Lancement de la requête d'alerte
# - avec une tolérance de 5 minutes sur l'horodatage (cf. section suivante)
# - qui exécute la fonction `traiter_batch_alertes` pour chaque batch
# - définissant un `checkpoint` pour la sauvegarde de l'état en cas de panne
# - la requête s'appelle `q_alertes_rupture`
# - avec un intervalle de traitement de 10 ssecondes
q_alertes = (
    stream_enrichi # ?
)

print("Requête d'alertes démarrée. En attente de données...")
time.sleep(60)   # laisser tourner 1 minute pour accumuler des événements

# Lecture des alertes produites
df_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
print(f"\nAlertes enregistrées : {df_alertes.count()}")
df_alertes.orderBy(F.desc("ts_alerte")).show(20, truncate=False)


---
## 2.7 Données tardives et watermark

Dans un système distribué, certains événements arrivent avec du retard
(réseau saturé, capteur hors ligne temporairement...). Spark doit décider
combien de temps attendre avant de considérer une fenêtre comme fermée.

C'est le rôle du **watermark** : il définit le retard maximal toléré.

```
Watermark de 5 minutes :
  Si le timestamp max observé est 10h30, Spark considère que
  toutes les données avant 10h25 sont arrivées.
  Les fenêtres fermées avant 10h25 ne seront plus mises à jour.
```

### Illustration avec des données tardives simulées


In [ ]:
import json, datetime

# Injection de données tardives :
# Écrire dans le répertoire de streaming un fichier avec des horodatages "dans le passé"
# Les dates sont dans le format ISO
# Le fichier est injecté dans le répertroie surveillé par Spark.
# Chaque line est homogène à une observation (ou snapshot).
# N.B. Ajouter ne migne dans le fichier de logs
def injecter_donnees_tardives(retard_minutes: int, nb_lignes: int = 20):
    """Écrit un fichier JSON avec des horodatages retardés.

    Args:
        retard_minutes : Retard à simuler (en minutes).
        nb_lignes      : Nombre de snapshots à générer.
    """
    # TODO

# Scénario 1 : retard de 3 min -- dans le watermark (5 min) -> données prises en compte
injecter_donnees_tardives(retard_minutes=3, nb_lignes=10)
time.sleep(15)

# Scénario 2 : retard de 12 min -- hors watermark -> données ignorées par les agrégations
injecter_donnees_tardives(retard_minutes=12, nb_lignes=10)
time.sleep(15)

print("\nStatut des requêtes actives :")
for q in spark.streams.active:
    prog = q.lastProgress
    print(f"  {q.name:<30} -- inputRows : {prog['numInputRows'] if prog else 'N/A'}")


In [ ]:
# [EXERCICE]
# Créez une nouvelle requête de streaming qui calcule,
# pour chaque arrondissement et sur une fenêtre basculante de 15 minutes,
# le MAXIMUM du taux_occupation observé.
# Écrivez le résultat en mode "update" vers la console.
#
# Rappel : fenêtre basculante = window(col, "15 minutes")  (sans 3e argument)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
q_exercice = None


---
## 2.8 Arrêt propre des requêtes et bilan

En production, les requêtes de streaming tournent indéfiniment.
Dans un contexte de cours, il faut les arrêter proprement pour libérer les ressources.


In [ ]:
# Arrêt de toutes les requêtes actives
actives = spark.streams.active
print(f"Requêtes actives à arrêter : {[q.name for q in actives]}")

# TODO : Arrêter proprement les requêtes et reseigner le fichier logs

print("\nToutes les requêtes de streaming sont arrêtées.")


In [ ]:
# Lecture finale des résultats accumulés
print("=== Résultats finaux ===")

print("\n-- Fenêtres arrondissement --")
try:
    df_fin_fenetres = spark.read.format("delta").load(
        str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
    )
    print(f"  {df_fin_fenetres.count()} fenêtres enregistrées")
    df_fin_fenetres.orderBy(F.desc("fenetre_debut")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucun résultat : {e}")

print("\n-- Alertes rupture --")
try:
    df_fin_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
    print(f"  {df_fin_alertes.count()} alertes enregistrées")
    df_fin_alertes.orderBy(F.desc("ts_alerte")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucune alerte : {e}")


---
## Bilan du Jour 2

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Vues temporaires et SQL de base | Spark SQL | `createOrReplaceTempView`, SQL natif |
| Questions métier analytiques | Spark SQL | `LEFT ANTI JOIN`, `CASE WHEN`, sous-requêtes |
| Fenêtrage analytique | Spark SQL / DataFrame | `OVER`, `LAG`, `LEAD`, `ROW_NUMBER`, `AVG OVER` |
| Écriture Delta Lake | Delta | `format("delta")`, partitionnement |
| Time-travel | Delta | `versionAsOf`, historique des opérations |
| Mise à jour incrémentale | Delta | `MERGE INTO`, `whenMatchedUpdateAll` |
| Source de streaming | Structured Streaming | `readStream`, schéma explicite, file source |
| Sink console | Structured Streaming | débogage, `outputMode("append")` |
| Fenêtres glissantes | Structured Streaming | `window()`, basculante vs glissante |
| Sink Delta | Structured Streaming | écriture transactionnelle en flux |
| Alertes avec `foreachBatch` | Structured Streaming | logique inter-batchs, état partagé |
| Watermark et late data | Structured Streaming | tolérance, fermeture de fenêtres |

### Points d'attention

- En `outputMode("append")`, Spark n'écrit que des lignes **nouvelles et définitives**.
  Pour les agrégations fenêtrées, cela implique un watermark : Spark attend que la fenêtre
  soit fermée avant d'émettre son résultat.
- `foreachBatch` est puissant mais introduit un état mutable (`etat_ruptures`) qui
  n'est **pas persisté dans le checkpoint**. En cas de redémarrage, l'état est perdu.
  Pour un système de production, il faudrait utiliser `mapGroupsWithState` ou
  stocker l'état dans Delta Lake.
- Le checkpoint est **obligatoire** pour toute requête qui ne doit pas repartir de zéro
  après un redémarrage.

### Pour demain (Jour 3)

La table Delta `disponibilite` (batch) et les résultats des fenêtres glissantes
(streaming) serviront de données d'entrée pour le Jour 3 : construction des features,
entraînement d'un modèle de régression avec MLlib, clustering des stations avec K-Means
et suivi des expériences avec MLflow.


In [ ]:
#spark.stop()
print("SparkSession arrêtée. À demain !")
